In [ ]:
# Wine Quality ML Pipeline - 환경 설정 및 데이터 로드
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 5)
sns.set_style('whitegrid')
%matplotlib inline

# UCI Wine Quality CSV URLs (red + white)
URL_RED = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
URL_WHITE = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

df_red = pd.read_csv(URL_RED, sep=';')
df_white = pd.read_csv(URL_WHITE, sep=';')
df_red['type'] = 'red'
df_white['type'] = 'white'
df = pd.concat([df_red, df_white], ignore_index=True)
print('Shape:', df.shape)
print(df.head())

In [ ]:
# EDA: 품질 분포, 결측치, 상관
print('결측치:', df.isnull().sum().sum())
print('\nquality 분포:')
print(df['quality'].value_counts().sort_index())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['quality'].hist(ax=axes[0], bins=11, edgecolor='black')
axes[0].set_title('Quality distribution')
sns.heatmap(df.drop(columns=['type']).corr(), ax=axes[1], cmap='coolwarm', center=0, fmt='.2f')
axes[1].set_title('Feature correlation')
plt.tight_layout()
plt.show()

In [ ]:
# 타깃: 연속 평점 (quality 0-10, 회귀)
# 3단계 분류 대신 평점 예측(회귀)
print('Target: quality (0-10), regression')
print('quality 분포:', df['quality'].describe())

In [ ]:
# Feature 순서 고정 (API/프론트와 동일하게 유지)
FEATURE_ORDER = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol'
]
# 레드 / 화이트 구분 (타입별로 별도 모델 학습)
df_red = df[df['type'] == 'red'].copy()
df_white = df[df['type'] == 'white'].copy()
print('Red:', df_red.shape[0], '건, White:', df_white.shape[0], '건')

In [ ]:
# Train/Test 분할 및 StandardScaler (레드·화이트 각각, 회귀)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

splits = {}   # wine_type -> (X_train_scaled, X_test_scaled, y_train, y_test)
scalers = {}  # wine_type -> StandardScaler

for wine_type, part in [('red', df_red), ('white', df_white)]:
    X = part[FEATURE_ORDER]
    y = part['quality']  # 연속값 0-10
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    scalers[wine_type] = scaler
    splits[wine_type] = (X_train_scaled, X_test_scaled, y_train, y_test)
    print(f'{wine_type}: Train {X_train.shape[0]}, Test {X_test.shape[0]}')

In [ ]:
# 모델 비교 (레드·화이트 각각, 회귀, R² 기준 best 선택)
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_factories = {
    'Ridge': lambda: Ridge(random_state=42),
    'Random Forest': lambda: RandomForestRegressor(n_estimators=100, random_state=42),
    'SVR': lambda: SVR(),
}
best_models = {}  # wine_type -> fitted model
best_names = {}   # wine_type -> model name

for wine_type in ['red', 'white']:
    X_train_s, X_test_s, y_train, y_test = splits[wine_type]
    results = []
    fitted = {}
    for name, factory in model_factories.items():
        m = factory()
        m.fit(X_train_s, y_train)
        fitted[name] = m
        y_pred = m.predict(X_test_s)
        results.append({
            'Model': name,
            'MSE': mean_squared_error(y_test, y_pred),
            'MAE': mean_absolute_error(y_test, y_pred),
            'R2': r2_score(y_test, y_pred)
        })
    results_df = pd.DataFrame(results)
    best_name = results_df.loc[results_df['R2'].idxmax(), 'Model']
    best_models[wine_type] = fitted[best_name]
    best_names[wine_type] = best_name
    print(f'=== {wine_type.upper()} ===')
    print(results_df)
    print(f'Best: {best_name}\n')

In [ ]:
# 최종 모델 및 아티팩트 저장 (models/red/, models/white/)
import joblib
import json

MODELS_BASE = Path('../models') if Path('.').resolve().name == 'notebooks' else Path('models')
MODELS_BASE = MODELS_BASE.resolve()
MODELS_BASE.mkdir(parents=True, exist_ok=True)

SAMPLE_NAMES = [
    '포르투갈 레드 클래식', '빈야드 선셋', '도우루 루지', '오크 배럴 리저브', '청풍 화이트',
    '선셋 블러쉬', '미뇨 레드', '그린 밸리 드라이', '알렌테주 레드', '비냐 베르데',
    '달빛 화이트', '이탈리아노 스타일', '코르크 오크', '리버사이드 로제', '골든 아워',
    '테라스 레드', '마르베드 리저브', '스프링 블로섬', '하베스트 문', '실버 드롭',
    '앰버 글라스', '크림슨 나이트', '로맹 스타일', '피노 그리지오', '캐스케이드 화이트',
    '선라이즈 스파클', '다크 체리 레드', '라벤더 힐', '피닉스 레드', '스타더스트 화이트',
    '올드 월드 블렌드', '뉴 월드 드라이', '세레나데 로제', '트와일라이트 레드', '모닝 듀',
    '테라코타', '바다 바람 화이트', '포레스트 그린', '와일드베리', '허니드 오크',
    '선셋 테라스', '문리트', '스톤 페블', '피오네', '그랑 크뤼 스타일',
    '캐비어 페어링', '치즈 페어링', '디너 레드', '아페리티프', '데저트 스위트',
    '리조트 화이트', '셀러스 셀렉션', '윈터 레드', '썸머 드라이', '오토먼 화이트',
    '노트르담', '리버뷰', '마운틴 에어', '비스트로 하우스', '캐슬 리저브',
    '로열 블렌드', '가든 파티', '테이스팅룸 스페셜', '바인야드 에디션', '리미티드 런',
    '레거시 레드', '레거시 화이트', '헤리티지 블렌드', '클래식 리저브', '시그니처',
    '소울 오브 포도', '테라의 선물', '포도밭 이야기', '오래된 숙성', '신선한 수확',
]

for wine_type in ['red', 'white']:
    out_dir = MODELS_BASE / wine_type
    out_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_models[wine_type], out_dir / 'model.joblib')
    joblib.dump(scalers[wine_type], out_dir / 'scaler.joblib')
    with open(out_dir / 'feature_order.json', 'w') as f:
        json.dump(FEATURE_ORDER, f, indent=2)
    part = df_red if wine_type == 'red' else df_white
    n_samples = min(40, len(part))
    samples_df = part[FEATURE_ORDER].sample(n=n_samples, random_state=42)
    samples_list = []
    for i, (_, row) in enumerate(samples_df.iterrows()):
        name = SAMPLE_NAMES[i % len(SAMPLE_NAMES)]
        if i >= len(SAMPLE_NAMES):
            name = f'{name} {1 + i // len(SAMPLE_NAMES)}'
        samples_list.append({'name': name, 'type': wine_type, **row.to_dict()})
    with open(out_dir / 'samples.json', 'w', encoding='utf-8') as f:
        json.dump(samples_list, f, ensure_ascii=False, indent=2)
    print(f'Saved {wine_type} → {out_dir} (model, scaler, feature_order, samples.json)')